In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, GridSearchCV, ParameterGrid, train_test_split, StratifiedShuffleSplit
import ml_utilities


In [2]:
TESTS_PATH = "./results/"
INFO_PATH = "./results/experiments_info"
DATASET_TEST_PATH = 'DBs/PenDigits/pendigits_te.txt'
DATASET_PATH = 'DBs/PenDigits/pendigits_tr.txt'

In [3]:
def read_data():
    feat_count = 16
    data_path = DATASET_PATH # Impostare il percorso corretto
    patterns, labels = ml_utilities.load_labeled_dataset_from_txt(data_path, feat_count)
    test_path = DATASET_TEST_PATH
    x_test = ml_utilities.load_unlabeled_dataset_from_txt(test_path, feature_count)
    return patterns, labels, x_test

In [4]:
from sklearn.preprocessing import StandardScaler


def normalize_data(x_train, x_test):
    scaler = StandardScaler()
    scaler.fit(dataset_patterns)
    x_train_normalized = scaler.fit_transform(x_train)
    x_test_normalized = scaler.fit_transform(x_test)

    return x_train_normalized, x_test_normalized

In [5]:
model_params = {
    'SVC': {
        'model': SVC( ),
        'params' : {
            'C': [2**i for i in range(-5,16)] + [1],  #Regularization parameter. Providing only two as SVM is slow
            'kernel': ['rbf','linear'],
            'gamma': [2**i for i in range(-5,16)] + [0.00009111627561154887],
            'class_weight':['balanced', None],
            'degree' : [1,2,3,4,5,6]
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params' : {
            'n_neighbors': [1,3,5,7,9,11,13,15,17,19,21,23,25,27],
            'weights': ['uniform','distance'],
            'algorithm': ["ball_tree","kd_tree","brute"],
            'metric': ["minkowski","euclidean","l1", "l2","manhattan"],

        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(solver='liblinear',multi_class='auto'),
        'params': {
            'C': [2**i for i in range(-5,16)] + [1],  #Regularization. . Providing only two as LR can be slow
            'penalty': ['l1', 'l2']
        }
    }
}
scores=[]

In [6]:
params = {
        'model': SVC(),
        'params' : {
            'C': [2**i for i in range(-5,16)],  #Regularization parameter. Providing only two as SVM is slow
            'kernel': ['rbf'],
            'gamma': [2**i for i in range(-15,4)] + [0.00009111627561154887],
        }
}
scores=[]

In [7]:
# Caricamento del dataset
feature_count = 16
dataset_path = 'DBs/PenDigits/pendigits_tr.txt'  # Impostare il percorso corretto

dataset_patterns, dataset_labels = ml_utilities.load_labeled_dataset_from_txt(dataset_path, feature_count)
print('Shape dataset:', dataset_patterns.shape)
print('Shape labels:', dataset_labels.shape)

Shape dataset: (442, 16)
Shape labels: (442,)


In [8]:
np.random.seed(42)

x_train, y_train, x_test = read_data()
x_train, x_test = normalize_data(x_train,x_test)
params = {
            'C': [2**i for i in range(-5,16)],  #Regularization parameter. Providing only two as SVM is slow
            'kernel': ['rbf'],
            'gamma': [2**i for i in range(-15,4)] + [0.00009111627561154887],

}
svc = SVC()
scores = []

In [9]:


for data_test_size in [0.3, 0.4, 0.5]:
    for folds in [9,10]:
        cross_val = StratifiedShuffleSplit(n_splits=folds, test_size=data_test_size, random_state=42)
        grid =  GridSearchCV(svc,
                             param_grid=params,
                             cv=cross_val)

        grid.fit(x_train,y_train)
        scores.append({
            'data_test_size': data_test_size,
            'folds': folds,
            'best_score': grid.best_score_,
            'mean_test_score': grid.cv_results_['mean_test_score'],
            'best_params': grid.best_params_,
        })
        print(f"fit completed")
    d9 = pd.DataFrame(scores,columns=['data_test_size','folds','best_score','mean_test_score','best_params'])
    print(f"data_test_size= {data_test_size} and folds = {folds}")
    print(f"fit completed")

fit completed
fit completed
data_test_size= 0.3 and folds = 10
fit completed
fit completed
fit completed
data_test_size= 0.4 and folds = 10
fit completed
fit completed
fit completed
data_test_size= 0.5 and folds = 10
fit completed


In [10]:
import ast

# Risultati dei 513 esperimenti (griglie SVC, k-NN e regressione logistica su vari split)
dataframe = pd.read_csv(TESTS_PATH + "Training_output.csv", index_col=0)
dataframe["best_params"] = dataframe["best_params"].map(ast.literal_eval)
dataframe.sort_values(by='best_score', ascending=False)
dataframe

,data_test_size,folds,model,best_score,mean_test_score,best_params
0,0.10,1,SVC,0.933333,"[0.15555555555555556, 0.3111111111111111, 0.15...","{'C': 0.125, 'class_weight': 'balanced', 'degr..."
486,0.05,1,SVC,0.913043,"[0.08695652173913043, 0.21739130434782608, 0.0...","{'C': 0.125, 'class_weight': 'balanced', 'degr..."
1,0.10,1,KNeighbors,0.888889,"[0.7555555555555555, 0.7555555555555555, 0.711...","{'algorithm': 'ball_tree', 'metric': 'minkowsk..."
507,0.05,8,SVC,0.880435,"[0.08695652173913043, 0.2663043478260869, 0.08...","{'C': 1, 'class_weight': 'balanced', 'degree':..."
9,0.10,4,SVC,0.877778,"[0.13888888888888887, 0.26111111111111107, 0.1...","{'C': 0.5, 'class_weight': 'balanced', 'degree..."
...,...,...,...,...,...,...
473,0.95,5,logistic_regression,0.611429,"[0.08571428571428572, 0.2680952380952381, 0.08...","{'C': 32, 'penalty': 'l2'}"
484,0.95,9,KNeighbors,0.607407,"[0.6074074074074074, 0.6074074074074074, 0.552...","{'algorithm': 'ball_tree', 'metric': 'minkowsk..."
482,0.95,8,logistic_regression,0.605060,"[0.08571428571428572, 0.28482142857142856, 0.0...","{'C': 16, 'penalty': 'l2'}"
485,0.95,9,logistic_regression,0.603968,"[0.08571428571428572, 0.2783068783068783, 0.08...","{'C': 16, 'penalty': 'l2'}"


In [11]:
dataframe['score'] = 0
# for SVC
datawork = dataframe.loc[dataframe['model'] == "SVC"]
dataframe['score'].update(datawork['best_params'].map(lambda x: SVC(**x).fit(x_train, y_train).score(x_train, y_train)))
# for KNN
datawork = dataframe.loc[dataframe['model'] == "KNeighbors"]
dataframe['score'].update(datawork['best_params'].map(lambda x: KNeighborsClassifier(**x).fit(x_train, y_train).score(x_train, y_train)))# for KNN
dataframe.sort_values(by='best_score', ascending=False)
datacopy = dataframe
datacopy = datacopy[(dataframe.data_test_size > 0.25) & (dataframe.data_test_size < 0.7)]
datacopy = datacopy[(dataframe.folds > 5)]
datacopy = datacopy[(dataframe.model != 'logistic_regression')]

/tmp/ipykernel_1988/2765147331.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  datacopy = datacopy[(dataframe.folds > 5)]
/tmp/ipykernel_1988/2765147331.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  datacopy = datacopy[(dataframe.model != 'logistic_regression')]


In [12]:
databest = d9.sort_values(by='best_score', ascending=True)
best_classifier = SVC(**databest['best_params'][0])
# Addestramento del classificatore
best_classifier.fit(x_train, y_train)
# Calcolo delle prediction
predictions = best_classifier.predict(x_test)
predictions

array([2., 4., 0., ..., 0., 8., 2.])

In [13]:
databest = d9.sort_values(by='best_score', ascending=True)
best_classifier = SVC(**databest['best_params'][0])
best_classifier

SVC(C=2, gamma=0.125)

In [14]:
result_path = 'Es3Predictions.txt'
with open(result_path, "w") as f:
    for prediction in predictions:
        f.write(str(int(prediction)) + '\n')
print('Salvataggio delle prediction riuscito')

Salvataggio delle prediction riuscito
